<a href="https://colab.research.google.com/github/awais2/RAG_Course/blob/main/RAG_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Retrieval-Augmented Generation (RAG)** is a method that integrates information retrieval to give generative language models additional information.

**A typical RAG pipeline comprises of 2 main components:**

1. a **Retriever Module** that first selects relevant documents or pieces of information from a large corpus based on the input query,
2. an **Answer Generation Module** that produces more accurate and contextually relevant responses.

#### Steps to implement a RAG pipeline (Part 1/2):
1. Indexing Documents

2. Creating Embeddings

3. Create a vector store and store embeddings

In [ ]:
pip install langchain langchain-community langchain-core langchain-openai langchain-text-splitters openai chromadb python-dotenv colorama

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 95.8 MB/s eta 0:00:0

In [ ]:
from dotenv import load_dotenv
from colorama import Fore
import warnings
warnings.filterwarnings("ignore")

load_dotenv()

False

1. Load and split documents

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter

def load_documents():
    """Load a file from path, split it into chunks, embed each chunk and load it into the vector store."""
    loader = TextLoader("user-manual.txt")
    raw_text = loader.load()
    text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=0, separator="\n\n")
    return text_splitter.split_documents(raw_text)

documents = load_documents()
print(f"Loaded {len(documents)} documents")

Loaded 8 documents


2. Create vector store and store embeddings

In [ ]:
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from openai import OpenAI
from google.colab import userdata

import os

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)

def get_embedding(text_to_embed):
    response = client.embeddings.create(
        model= "text-embedding-ada-002",
        input=[text_to_embed]
    )
    print(response.data[0].embedding)

def load_embeddings(user_query, documents):
    """Create a vector store from a set of documents."""
    embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)
    db = Chroma.from_documents(documents, embeddings)
    get_embedding(user_query)
    _ = [get_embedding(doc.page_content) for doc in documents]
    return db.as_retriever()

retriever = load_embeddings("I have an error code E2", documents)

[-0.01569574698805809, -0.0038961321115493774, -0.01547330990433693, -0.023967642337083817, -0.02840249054133892, 0.014173440635204315, -0.023230817168951035, -0.010259930044412613, -0.0037119260523468256, -0.008112017996609211, 0.00795214157551527, 0.0031888503581285477, 0.0014971087221056223, -0.005265512969344854, -0.01872645877301693, 0.00029086312861181796, 0.006419407669454813, 0.013707712292671204, 0.01269284076988697, -0.005790326744318008, -0.004049058072268963, 0.0016005074139684439, -0.013860637322068214, -0.006846904754638672, 0.005964105948805809, -0.014402829110622406, 0.0032375084701925516, -0.017461344599723816, -0.016376962885260582, -0.014778192155063152, 0.01269284076988697, -0.026914939284324646, -0.022980576381087303, -0.04368116706609726, -0.0281661506742239, -0.004935332573950291, -0.020019376650452614, 0.0016005074139684439, 0.02204911783337593, -0.011351264081895351, 0.048491377383470535, 0.013151617720723152, -0.010816024616360664, -0.01076736580580473, -0.020

#### Steps to implement a RAG pipeline (Part 2/2):
1. Define a prompt

2. Create and run the retrieval chain


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts.chat import (
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)
from langchain.prompts import ChatPromptTemplate, PromptTemplate

prompt: str = "You are a customer support specialist who answers questions {question} and assist users with general inquiries"
prompt_template = PromptTemplate.from_template(prompt)

template: str = """/
    You are a customer support specialist /
    question: {question}. You assist users with general inquiries based on {context} /
    and  technical issues. /
    """
system_message_prompt = SystemMessagePromptTemplate.from_template(template)
human_message_prompt = HumanMessagePromptTemplate.from_template(
    input_variables=["question", "context"],
    template="{question}",
)
chat_prompt_template = ChatPromptTemplate.from_messages(
    [system_message_prompt, human_message_prompt]
)

model = ChatOpenAI(api_key=OPENAI_API_KEY)

2. Create and run the chain

**without RAG**


In [ ]:
from langchain.schema import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough

def generate_response(retriever, query):
    """Generate a response using the retriever and the query."""
    # Create a prompt template using a template from the config module and input variables
    # representing the context and question.
    # create the prompt
    chain = (
        {"question": RunnablePassthrough()}
        | prompt_template
        | model
        | StrOutputParser()
    )
    return chain.invoke(query)

response = generate_response(retriever, "I have an error code E2")
print(f"{Fore.GREEN}{response}{Fore.RESET}")

Hello! I'm sorry to hear that you're experiencing error code E2. This error code usually indicates a problem with the voltage supply or power source. Here are some steps you can try to resolve this issue:

1. Check the power source: Ensure that the appliance is properly plugged in and receiving power. Try plugging it into a different outlet or resetting the circuit breaker if needed.

2. Check the voltage supply: Make sure that the voltage supply is within the range specified in the appliance's user manual. If the voltage supply is too high or too low, it can cause error code E2.

3. Contact the manufacturer: If the issue persists after trying the above steps, it's best to contact the manufacturer's customer support team for further assistance. They may be able to provide specific troubleshooting steps or arrange for a technician to inspect and repair the appliance.

If you have any other questions or need assistance with general inquiries, feel free to let me know!


**with RAG**

In [ ]:
from langchain.schema import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough

def generate_response(retriever, query):
    """Generate a response using the retriever and the query."""
    # Create a prompt template using a template from the config module and input variables
    # representing the context and question.
    # create the prompt
    chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | chat_prompt_template
        | model
        | StrOutputParser()
    )
    return chain.invoke(query)

response = generate_response(retriever, "I have an error code E3")
print(f"{Fore.GREEN}{response}{Fore.RESET}")

Error code E3 indicates a drum rotation problem with your washing machine. To troubleshoot this issue, please check the laundry load for balance. Make sure that the load inside the washing machine is distributed evenly to avoid imbalance during the rotation cycle. Once you have balanced the laundry load, try running the washing machine again to see if the error persists. If the problem continues, please contact our after-sales service for further assistance. You can reach them at the number provided below:

Phone: +33 880 880 887

You can also visit our website at www.machinelaverxyz.com or send an email to serviceclient@machinelaverxyz.com for additional support. Thank you for choosing Washing Machine XYZ, and we are here to help you resolve this issue.
